<a href="https://colab.research.google.com/github/fernandodeeke/Hydraulics/blob/main/closed_two_tank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import numpy as np
import matplotlib.pyplot as plt

# ----- physical constant --------------------------------------------
G = 9.81                            # gravitational acceleration [m/s^2]

# ----- colours / styles ---------------------------------------------
C1, C2 = "#1f77b4", "#d62728"       # tank 1 (h1), tank 2 (h2)
CEQ, CTAU, CTEQ = "#555555", "#7f7f7f", "#9467bd"

# init values (also used by the Reset button)
INIT = dict(A1=1.5, A2=1.0, h10=3.0, h20=0.5, R12=1.0, Cd=0.60, a=0.20, T=20.0)
N_T = 800


# ====================================================================
#  Exact analytic solutions of both models
# ====================================================================
def solutions(t, A1, A2, h10, h20, R12, k12):
    Asum  = A1 + A2
    hstar = (A1 * h10 + A2 * h20) / Asum
    D0    = h10 - h20

    tau   = R12 * A1 * A2 / Asum                 # linear time constant
    D_lin = D0 * np.exp(-t / tau)

    gamma = k12 * Asum / (A1 * A2)               # turbulent
    t_eq  = 2.0 * np.sqrt(abs(D0)) / gamma if (D0 != 0 and k12 > 0) else 0.0
    base  = np.sqrt(abs(D0)) - 0.5 * gamma * t
    D_tur = np.sign(D0) * np.maximum(base, 0.0) ** 2

    f1, f2 = A2 / Asum, A1 / Asum
    return {
        "h1_lin": hstar + f1 * D_lin, "h2_lin": hstar - f2 * D_lin,
        "h1_tur": hstar + f1 * D_tur, "h2_tur": hstar - f2 * D_tur,
        "hstar": hstar, "tau": tau, "t_eq": t_eq,
    }


# ====================================================================
#  Build a single figure for a given parameter set
# ====================================================================
def build_figure(A1, A2, h10, h20, R12, Cd, a, T):
    k12 = Cd * a * np.sqrt(2.0 * G)              # turbulent conductance
    t = np.linspace(0.0, T, N_T)
    s = solutions(t, A1, A2, h10, h20, R12, k12)

    # ADDED dpi=300 HERE TO INCREASE PLOT RESOLUTION
    fig, ax = plt.subplots(figsize=(6.5, 3.0), dpi=150)

    ax.plot(t, s["h1_lin"], color=C1, lw=2., ls="-",  label=r"$h_1$  linear")
    ax.plot(t, s["h2_lin"], color=C2, lw=2., ls="-",  label=r"$h_2$  linear")
    ax.plot(t, s["h1_tur"], color=C1, lw=2., ls="--", label=r"$h_1$  turbulent")
    ax.plot(t, s["h2_tur"], color=C2, lw=2., ls="--", label=r"$h_2$  turbulent")
    ax.axhline(s["hstar"], color=CEQ,  ls=":",  lw=1.0, label=r"$h^*$")
    ax.axvline(s["tau"],   color=CTAU, ls=":",  lw=1.0, label=r"$\tau$ (linear)")
    ax.axvline(s["t_eq"],  color=CTEQ, ls="-.", lw=1.0, label=r"$t_{\rm eq}$ (turbulent)")

    ymax = max(h10, h20, s["hstar"]) * 1.12 + 0.05
    ax.set_xlim(0, T)
    ax.set_ylim(-0.02, ymax)
    ax.set_xlabel("$t$ (s)")
    ax.set_ylabel("water height (m)")
    ax.tick_params(axis='both', labelsize=8)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", framealpha=0.92, fontsize=7, ncol=2)
    ax.set_title(
        r"Linear (solid) vs. turbulent (dashed):   "
        r"$k_{12}=C_d\,a\sqrt{2g}=%.3f$,   $\tau=%.2f$,   $t_{\rm eq}=%.2f$"
        % (k12, s["tau"], s["t_eq"]), fontsize=8)
    fig.tight_layout()
    return fig


# ====================================================================
#  Interactive launcher (ipywidgets -> works in Colab)
# ====================================================================
def launch():
    import ipywidgets as widgets
    from IPython.display import display

    # Enables crisp rendering in Jupyter/Colab environments
    from IPython.display import set_matplotlib_formats
    set_matplotlib_formats('retina')

    try:
        from google.colab import output as _co
        _co.enable_custom_widget_manager()
    except Exception:
        pass

    style  = {"description_width": "150px"}
    layout = widgets.Layout(width="380px")

    def fslider(desc, lo, hi, step, val, fmt=".2f"):
        return widgets.FloatSlider(description=desc, min=lo, max=hi, step=step,
                                   value=val, continuous_update=True,
                                   style=style, layout=layout, readout_format=fmt)

    sl = {
        "A1":  fslider("A\u2081  (tank 1 area)",   0.2,  5.0,  0.05, INIT["A1"]),
        "A2":  fslider("A\u2082  (tank 2 area)",   0.2,  5.0,  0.05, INIT["A2"]),
        "h10": fslider("h\u2081(0)  init. level",  0.0,  5.0,  0.05, INIT["h10"]),
        "h20": fslider("h\u2082(0)  init. level",  0.0,  5.0,  0.05, INIT["h20"]),
        "R12": fslider("R\u2081\u2082  (linear)",  0.1,  5.0,  0.05, INIT["R12"]),
        "Cd":  fslider("C_d  (discharge coeff.)",  0.05, 1.0,  0.01, INIT["Cd"]),
        "a":   fslider("a  (valve area)",          0.01, 1.0,  0.01, INIT["a"], fmt=".3f"),
        "T":   fslider("t_max  (sim. time)",       1.0,  60.0, 1.0,  INIT["T"]),
    }

    def plot(A1, A2, h10, h20, R12, Cd, a, T):
        fig = build_figure(A1, A2, h10, h20, R12, Cd, a, T)
        plt.show()
        plt.close(fig)

    out = widgets.interactive_output(plot, sl)

    reset = widgets.Button(description="Reset")
    def _reset(_):
        for k, w in sl.items():
            w.value = INIT[k]
    reset.on_click(_reset)

    controls = widgets.HBox([
        widgets.VBox([sl["A1"], sl["A2"], sl["h10"], sl["h20"]]),
        widgets.VBox([sl["R12"], sl["Cd"], sl["a"], sl["T"]]),
    ])
    caption = widgets.HTML(
        "<div style='color:#555;font-size:60%'>Colour = tank "
        "(h&#8321; blue, h&#8322; red); line style = model "
        "(solid linear, dashed turbulent). The turbulent conductance is "
        "k&#8321;&#8322; = C<sub>d</sub>&#183;a&#183;&#8730;(2g), g = 9.81. "
        "Both models share the same h*.</div>")
    display(controls, reset, caption, out)


if __name__ == "__main__":
    launch()

/tmp/ipykernel_1848/3678472035.py:84: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


Button(description='Reset', style=ButtonStyle())

HTML(value="<div style='color:#555;font-size:60%'>Colour = tank (h&#8321; blue, h&#8322; red); line style = mo…

Output()